In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten

In [19]:
loaded_data = np.load('kinematic_dataset.npz')

In [20]:
print(loaded_data.files)

['X_train', 'X_test', 'Y_train', 'Y_test']


In [21]:
X_train = loaded_data['X_train']
X_test = loaded_data['X_test']
Y_train = loaded_data['Y_train']
Y_test = loaded_data['Y_test']

In [22]:
model = Sequential()

In [23]:
model.add(Flatten(input_shape = (40, 6)))
model.add(Dense(32, activation = 'relu'))
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

C:\Anaconda\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 240)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           7,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,745 (30.25 KB)

 Trainable params: 7,745 (30.25 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.compile(
    optimizer= 'adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')]
)

In [30]:
history = model.fit(
    X_train, Y_train,
    batch_size = 32,
    epochs = 30,
    validation_data = (X_test, Y_test)
)

Epoch 1/30


C:\Anaconda\Lib\site-packages\keras\src\backend\tensorflow\nn.py:789: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 445ms/step - accuracy: 0.7059 - loss: 0.6273 - val_accuracy: 0.8462 - val_loss: 0.5096
Epoch 2/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.7843 - loss: 0.4774 - val_accuracy: 0.8462 - val_loss: 0.3894
Epoch 3/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.9216 - loss: 0.3662 - val_accuracy: 1.0000 - val_loss: 0.2971
Epoch 4/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.9608 - loss: 0.2797 - val_accuracy: 1.0000 - val_loss: 0.2291
Epoch 5/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.9804 - loss: 0.2106 - val_accuracy: 1.0000 - val_loss: 0.1789
Epoch 6/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 1.0000 - loss: 0.1651 - val_accuracy: 1.0000 - val_loss: 0.1399
Epoch 7/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 1.0000 - loss: 0.1287 - val_accuracy: 1.0000 - val_loss: 0.1104
Epoch 8/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 1.0000 - loss: 0.1037 - val_accuracy: 1.0000 - val_loss: 0.0881
Epo

In [32]:
#converting and compressing for esp32
converter = tf.lite.TFLiteConverter.from_keras_model(model)

In [33]:
tflite_model = converter.convert()

INFO:tensorflow:Assets written to: C:\Users\VAIBHA~1\AppData\Local\Temp\tmp43gd5j60\assets


INFO:tensorflow:Assets written to: C:\Users\VAIBHA~1\AppData\Local\Temp\tmp43gd5j60\assets


Saved artifact at 'C:\Users\VAIBHA~1\AppData\Local\Temp\tmp43gd5j60'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2271919171216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2271919172176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2271919171408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2271919172944: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [35]:
with open('Gesture_model.tflite', 'wb') as file:
    file.write(tflite_model)

print('Model successfully saved')

Model successfully saved
